In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
import os
from pathlib import Path

import numpy as np

In [4]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
LAYER = 11
DATA_DIR = os.environ["DATA_DIR"]

In [5]:
from reliable_monitoring.dataset import ActivationConfig

activation_config = ActivationConfig(model_name=MODEL_NAME, layer=LAYER)

/Users/ep/Documents/research/vrcp_proj/reliable-llm-monitoring/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Cascade

In [6]:
from reliable_monitoring.dataset import load_dataset, sample_from_dataset

train_dataset = sample_from_dataset(
    load_dataset(
        Path(f"{DATA_DIR}/training/prompts_4x/train.jsonl"),
        activation_config=activation_config,
    ),
    100,
)

# calibration and test are both exchangeable (actually i.i.d.)
calibration_dataset = load_dataset(
    # Path(f"{DATA_DIR}/evals/dev/toolace_balanced_apr_22.jsonl"),
    Path(f"{DATA_DIR}/evals/dev/anthropic_balanced_apr_23.jsonl"),
    activation_config=activation_config,
)
calibration_dataset = sample_from_dataset(
    calibration_dataset,
    100,
)

test_dataset = load_dataset(
    # Path(f"{DATA_DIR}/evals/test/toolace_test_balanced_apr_22.jsonl"),
    Path(f"{DATA_DIR}/evals/test/anthropic_test_balanced_apr_23.jsonl"),
    activation_config=activation_config,
)
test_dataset = sample_from_dataset(
    test_dataset,
    100,
)

In [7]:
from reliable_monitoring.dataset import split_dataset

opt_dataset, calibration_dataset = split_dataset(calibration_dataset, [0.5, 0.5])

In [8]:
from reliable_monitoring.probes import SequenceProbe

probe = SequenceProbe(reduction_strategy="mean")
probe.fit(train_dataset)

Reducing activations: 100%|██████████| 1/1 [00:00<00:00, 15.39it/s]


In [9]:
# run cascade evaluation

from reliable_monitoring.cascade import run_llm_baseline, run_offline_cascade
from reliable_monitoring.risks import (
    baseline_budget_cost,
    empirical_accuracy,
)

# Compute probe scores and baseline scores separately
print("Computing probe scores...")
probe_scores = probe.predict(opt_dataset)

print("Computing baseline scores...")
baseline_scores = run_llm_baseline(
    baseline_model_name=MODEL_NAME,
    dataset=opt_dataset,
    baseline_batch_size=16,
)

thresholds = np.linspace(0, 1, 11)[1:-1]  # Exclude 0 and 1
budget_scores = []
empirical_performance_scores = []

for i, t in enumerate(thresholds):
    print(f"Evaluating threshold: {t:.2f}, {i} of {len(thresholds)}")
    opt_scores = run_offline_cascade(
        probe_scores,
        baseline_scores,
        threshold=t,
        merge_strategy="avg",
    )
    budget_scores.append(baseline_budget_cost(opt_scores))
    empirical_performance_scores.append(empirical_accuracy(opt_scores, opt_dataset))

Computing probe scores...


Reducing activations: 100%|██████████| 1/1 [00:00<00:00, 23.92it/s]


Computing baseline scores...


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
`torch_dtype` is deprecated! Use `dtype` instead!
100%|██████████| 7/7 [01:06<00:00,  9.54s/it]


Evaluating threshold: 0.10, 0 of 9
Evaluating threshold: 0.20, 1 of 9
Evaluating threshold: 0.30, 2 of 9
Evaluating threshold: 0.40, 3 of 9
Evaluating threshold: 0.50, 4 of 9
Evaluating threshold: 0.60, 5 of 9
Evaluating threshold: 0.70, 6 of 9
Evaluating threshold: 0.80, 7 of 9
Evaluating threshold: 0.90, 8 of 9


In [16]:
from reliable_monitoring.ltt_utils import is_pareto

costs = np.concatenate(
    [
        -np.array(budget_scores).reshape(-1, 1),  # we want to minimise the budget cost
        np.array(empirical_performance_scores).reshape(-1, 1),
    ],
    axis=1,
)


pareto_front = is_pareto(-costs)
pareto_front

array([ True,  True,  True,  True,  True, False, False,  True, False])

In [17]:
costs

array([[-0.  ,  0.52],
       [-0.  ,  0.52],
       [-0.  ,  0.52],
       [-0.  ,  0.52],
       [-0.  ,  0.52],
       [-0.04,  0.52],
       [-0.08,  0.52],
       [-0.14,  0.54],
       [-0.34,  0.54]])

In [18]:
import plotly.express as px

fig = px.scatter(
    x=-costs[:, 0],  # Budget cost (negative because we minimized -cost)
    y=costs[:, 1],  # Empirical performance
    title="Budget Cost vs Empirical Performance",
    labels={"x": "Budget Cost", "y": "Empirical Performance"},
)
fig.add_scatter(
    x=-costs[pareto_front, 0],
    y=costs[pareto_front, 1],
    mode="markers",
    marker=dict(color="red", size=10, symbol="star"),
    name="Pareto Front",
)
fig.show()

In [13]:
budget_scores

[0.0, 0.0, 0.0, 0.0, 0.0, 0.04, 0.08, 0.14, 0.34]

In [14]:
empirical_performance_scores

[0.52, 0.52, 0.52, 0.52, 0.52, 0.52, 0.52, 0.54, 0.54]

In [ ]:
# now define the empirical risk functions that we will use to find the pareto front and run LTT

# LTT

In [ ]:
# given train, validation, calibration and test splits
# train probes on `train`
# e2e score pipeline: give validation dataset and threshold, get final performance metrics (cost + score)
# vanilla threshold estimation using calibration set once
# LTT using above e2e pipeline and calibration set + pareto frontier etc...
# comapre e2e performance distributions across multiple runs

In [ ]:
# make e2e pipeline with final performance score + cost as a function of dataset splits
# find pareto front
# run fixed sequence testing